# 🗺️ Geospatial Demand Visualization

Decodes the anonymised geohash locations to latitude/longitude and visualizes
the spatial distribution of traffic demand across the city grid.

**Key insight:** demand is not uniformly distributed — it clusters in hotspots
that follow the urban road network, and these spatial patterns are stable
across time slots, which is why geohash×time target encodings are the
dominant predictive feature.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
print('train', train.shape, '| test', test.shape)

## Geohash → Lat/Lon decoder

In [ ]:
_B32 = '0123456789bcdefghjkmnpqrstuvwxyz'
_DEC = {c: i for i, c in enumerate(_B32)}

def decode_geohash(gh: str) -> tuple[float, float]:
    """Decode a geohash6 string to (latitude, longitude)."""
    lat, lon, is_lon = [-90.0, 90.0], [-180.0, 180.0], True
    for ch in str(gh).lower():
        cd = _DEC.get(ch)
        if cd is None:
            continue
        for mask in (16, 8, 4, 2, 1):
            if is_lon:
                mid = (lon[0] + lon[1]) / 2
                lon[0 if cd & mask else 1] = mid
            else:
                mid = (lat[0] + lat[1]) / 2
                lat[0 if cd & mask else 1] = mid
            is_lon = not is_lon
    return (lat[0] + lat[1]) / 2, (lon[0] + lon[1]) / 2

# decode all unique geohashes
all_gh = pd.concat([train['geohash'], test['geohash']]).unique()
coords = pd.DataFrame([decode_geohash(g) for g in all_gh], columns=['lat', 'lon'], index=all_gh)
coords.index.name = 'geohash'
print(f'{len(coords)} unique locations decoded')
print(f'lat: {coords.lat.min():.4f} to {coords.lat.max():.4f}')
print(f'lon: {coords.lon.min():.4f} to {coords.lon.max():.4f}')
coords.head()

## 1 · Spatial distribution of locations
Each dot is one geohash cell in the city grid. Note the clear urban structure —
locations are not randomly scattered but follow road corridors.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(coords['lon'], coords['lat'], s=8, alpha=0.5, c='#58a6ff', edgecolors='none')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Geohash Grid — All Decoded Locations', fontsize=14, fontweight='bold')
ax.set_facecolor('#0d1117')
fig.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.xaxis.label.set_color('#c9d1d9')
ax.yaxis.label.set_color('#c9d1d9')
ax.title.set_color('#e6edf3')
for spine in ax.spines.values(): spine.set_color('#30363d')
plt.tight_layout()
plt.savefig('../assets/geospatial_grid.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 2 · Demand hotspot map
Average demand per location across the full training period. Brighter = higher demand.
The spatial clustering of hotspots is the signal the geo-features capture.

In [ ]:
gh_demand = train.groupby('geohash')['demand'].mean()
plot_df = coords.join(gh_demand, how='inner')

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(plot_df['lon'], plot_df['lat'], c=plot_df['demand'],
                cmap='hot', s=15, alpha=0.8, edgecolors='none',
                norm=mcolors.PowerNorm(gamma=0.5))
cbar = plt.colorbar(sc, ax=ax, shrink=0.7, label='Mean demand')
cbar.ax.yaxis.label.set_color('#c9d1d9')
cbar.ax.tick_params(colors='#8b949e')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Demand Hotspot Map — Average Demand by Location', fontsize=14, fontweight='bold')
ax.set_facecolor('#0d1117')
fig.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.xaxis.label.set_color('#c9d1d9')
ax.yaxis.label.set_color('#c9d1d9')
ax.title.set_color('#e6edf3')
for spine in ax.spines.values(): spine.set_color('#30363d')
plt.tight_layout()
plt.savefig('../assets/demand_hotspot_map.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 3 · Demand by time of day (spatial evolution)
How the demand landscape shifts across the day. Each panel shows the same city grid at a
different hour — watch the hotspots activate as morning commuters appear.

In [ ]:
def mod(t):
    h, m = str(t).split(':')
    return int(h) * 60 + int(m)

train['hour'] = train['timestamp'].map(mod) // 60

hours = [0, 4, 8, 12, 16, 20]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.set_facecolor('#0d1117')
fig.suptitle('Spatial Demand Evolution by Hour', fontsize=16, fontweight='bold', color='#e6edf3', y=0.98)

for ax, h in zip(axes.flat, hours):
    hd = train[train['hour'] == h].groupby('geohash')['demand'].mean()
    pdf = coords.join(hd, how='inner')
    if len(pdf) == 0:
        ax.text(0.5, 0.5, f'{h:02d}:00\nno data', transform=ax.transAxes,
                ha='center', va='center', color='#8b949e', fontsize=12)
    else:
        ax.scatter(pdf['lon'], pdf['lat'], c=pdf['demand'], cmap='hot',
                   s=8, alpha=0.8, edgecolors='none',
                   norm=mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=train['demand'].quantile(0.95)))
    ax.set_title(f'{h:02d}:00', fontsize=13, fontweight='bold', color='#c9d1d9')
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#8b949e', labelsize=8)
    for spine in ax.spines.values(): spine.set_color('#30363d')

plt.tight_layout()
plt.savefig('../assets/demand_by_hour.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4 · Geohash clustering (K-means on coordinates)
The geo-clusters used as a feature in the model. Nearby locations get the same cluster
label, enabling the model to share demand patterns across adjacent cells.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=30, random_state=42, n_init=10).fit(coords[['lat', 'lon']])
coords['cluster'] = km.labels_

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(coords['lon'], coords['lat'], c=coords['cluster'],
                cmap='tab20', s=15, alpha=0.8, edgecolors='none')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('K-Means Geo-Clusters (k=30) — Spatial Feature', fontsize=14, fontweight='bold')
ax.set_facecolor('#0d1117')
fig.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.xaxis.label.set_color('#c9d1d9')
ax.yaxis.label.set_color('#c9d1d9')
ax.title.set_color('#e6edf3')
for spine in ax.spines.values(): spine.set_color('#30363d')
plt.tight_layout()
plt.savefig('../assets/geo_clusters.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Cluster sizes: min={coords.cluster.value_counts().min()}, max={coords.cluster.value_counts().max()}, median={coords.cluster.value_counts().median():.0f}')

## 5 · Neighbour demand spillover
Visualizes the spatial-neighbour feature: for each location, the mean demand of its
6 nearest geohash neighbours. The smooth gradient confirms that geohash adjacency
is preserved (the FAQ noted this) and nearby areas share demand levels.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=7).fit(coords[['lat', 'lon']])
_, idx = nn.kneighbors(coords[['lat', 'lon']])

gh_dem = train.groupby('geohash')['demand'].mean()
coords_plot = coords.join(gh_dem, how='inner')
nb_mean = []
for i, g in enumerate(coords_plot.index):
    neighbours = [coords.index[j] for j in idx[coords.index.get_loc(g)][1:] if coords.index[j] in gh_dem.index]
    nb_mean.append(np.mean([gh_dem[n] for n in neighbours]) if neighbours else gh_dem.get(g, 0))
coords_plot['nb_demand'] = nb_mean

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.set_facecolor('#0d1117')
for ax, col, title in zip(axes, ['demand', 'nb_demand'],
    ['Own demand', 'Neighbour mean demand (spatial spillover)']):
    sc = ax.scatter(coords_plot['lon'], coords_plot['lat'], c=coords_plot[col],
                    cmap='hot', s=12, alpha=0.8, edgecolors='none',
                    norm=mcolors.PowerNorm(gamma=0.5))
    plt.colorbar(sc, ax=ax, shrink=0.7)
    ax.set_title(title, fontsize=13, fontweight='bold', color='#c9d1d9')
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values(): spine.set_color('#30363d')
plt.tight_layout()
plt.savefig('../assets/spatial_spillover.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
### Key takeaways for the model
1. **Demand is spatially clustered** — hotspots are stable across hours, justifying
   per-geohash target encodings as the core feature.
2. **Geohash adjacency is preserved** (despite anonymisation) — the smooth spatial
   gradient in the neighbour plot confirms the FAQ's note and validates the
   `add_spatial_neighbour()` feature.
3. **Temporal evolution is gradual** — the same hotspots intensify/fade across hours
   rather than jumping randomly, which is why the denoised day-48 time-of-day
   profile transfers well to day-49 predictions.